# Exploratory Data Analysis: GDELT "Social Media" News Slice

This notebook analyzes the GDELT export used for the **"Perspectiva" for News Sentiment** project.

It includes:

- **fast iteration** on a random sample of `10,000` rows
- **memory-aware cleaning** of GDELT string fields
- **plots that guide the final globe design**

<div class="alert alert-info">
Each chart has a short note before it and a short summary after it.
</div>


## Dataset

The source data comes from the **GDELT Global Knowledge Graph**, a large database of worldwide news coverage with article metadata, themes, tones, and locations. The raw fields still need preprocessing before they can be used directly in plots.

For this export, the CSV contains `86,393` rows and six columns:

- `DATE`: a BigQuery-style integer timestamp (`YYYYMMDDHHMMSS`)
- `V2Tone`: a comma-separated vector whose **first value** is the article-level document tone
- `V2Locations`: a semicolon-delimited list of GDELT locations with embedded country codes
- `DocumentIdentifier`: the article URL
- `SourceCommonName`: the publisher domain
- `Themes`: a semicolon-delimited set of GDELT themes

**Data quality assessment**

- The raw file uses compact, analysis-unfriendly encodings for time, tone, and geography.
- `V2Locations` is the noisiest field: `17,108` rows are missing it entirely, and valid entries still require custom parsing.
- `Themes` has `1,414` missing values, so theme-based analysis must tolerate blanks.
- The full file occupies roughly **122.4 MB in memory-aware pandas form**, which is manageable but still large enough to justify sampling for rapid EDA.

<div class="alert alert-warning">
GDELT needs preprocessing before analysis.[cite: 7]
</div>


# Part 2: Setup, Data Loading, and Rapid Sampling

This section loads only the needed columns and then creates a **fixed random sample** for faster analysis.


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from urllib.parse import unquote, urlparse
import os
import re
import tempfile
import unicodedata
import warnings

import numpy as np
import pandas as pd

# Use a writable Matplotlib cache directory.
mpl_config_dir = Path(tempfile.gettempdir()) / "perspectiva_mpl_config"
mpl_config_dir.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_config_dir))

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import networkx as nx

from IPython.display import HTML, display
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

try:
    import nltk
    from nltk.corpus import stopwords
    NLTK_AVAILABLE = True
except ImportError:
    nltk = None
    stopwords = None
    NLTK_AVAILABLE = False

warnings.filterwarnings("ignore")
sns.set_theme(style="darkgrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)


The code finds the CSV automatically and then creates a fixed sample of `10,000` rows.

<div class="alert alert-info">
The sample is used for faster analysis. The random seed is fixed (`42`), so every rerun uses the same rows.
</div>


In [ ]:
def detect_string_dtype() -> str:
    # Use Arrow-backed strings when available.
    try:
        pd.Series(["ok"], dtype="string[pyarrow]")
        return "string[pyarrow]"
    except (TypeError, ImportError, ValueError):
        return "string"


def locate_data_file(project_root: Path) -> Path:
    # Find the newest GDELT CSV export.
    candidates = sorted(project_root.glob("bq-results-*.csv"))
    if not candidates:
        candidates = sorted(project_root.parent.glob("bq-results-*.csv"))
    if not candidates:
        raise FileNotFoundError("Could not find a file matching 'bq-results-*.csv'.")
    return max(candidates, key=lambda item: item.stat().st_mtime)


project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
data_path = locate_data_file(project_root)
string_dtype = detect_string_dtype()

dtype_map = {
    "DATE": "int64",
    "SourceCommonName": string_dtype,
    "DocumentIdentifier": string_dtype,
    "V2Locations": string_dtype,
    "V2Tone": string_dtype,
    "Themes": string_dtype,
}

df_raw = pd.read_csv(
    data_path,
    usecols=dtype_map.keys(),
    dtype=dtype_map,
    low_memory=False,
)

df = df_raw.sample(n=min(10_000, len(df_raw)), random_state=42).copy()

print(f"Loaded file: {data_path}")
print(f"Raw shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
print(f"Working sample: {df.shape[0]:,} rows")
print(f"Raw date range: {df_raw['DATE'].min()} -> {df_raw['DATE'].max()}")
print(f"Using string dtype backend: {string_dtype}")


**Sampling note.** The full dataset stays in `df_raw`, but the EDA below uses the sampled frame `df`.


# Part 3: Data Cleaning and Preprocessing

This step converts the main GDELT fields into usable columns:

- `DATE` -> `datetime`
- `V2Tone` -> scalar `sentiment_score`
- `V2Locations` -> primary `country_code` plus `country_name`
- `DocumentIdentifier` -> clean URL slug for lexical analysis
- `Themes` -> list of atomic theme labels


In [ ]:
GENERIC_COUNTRY_NAMES = {
    "republic of",
    "kingdom of",
    "state of",
    "province of",
    "district of",
    "city of",
    "federation of",
    "federal republic of",
    "democratic republic of",
    "general",
}

# Many GDELT country codes are not ISO-2, so common cases are mapped here.
GDELT_COUNTRY_OVERRIDES = {
    "AS": "Australia",
    "AU": "Austria",
    "BF": "Bahamas",
    "BO": "Belarus",
    "BX": "Brunei",
    "CB": "Cambodia",
    "CD": "Chad",
    "CF": "Congo (Kinshasa)",
    "CG": "Congo (Brazzaville)",
    "CH": "China",
    "DA": "Denmark",
    "EI": "Ireland",
    "EN": "Estonia",
    "EZ": "Czechia",
    "FR": "France",
    "GM": "Germany",
    "GR": "Greece",
    "HU": "Hungary",
    "IS": "Israel",
    "IT": "Italy",
    "IV": "Cote d'Ivoire",
    "JA": "Japan",
    "KE": "Kenya",
    "KN": "North Korea",
    "KS": "South Korea",
    "LA": "Laos",
    "LG": "Latvia",
    "LH": "Lithuania",
    "LO": "Slovakia",
    "LS": "Liechtenstein",
    "LU": "Luxembourg",
    "MI": "Malawi",
    "MO": "Morocco",
    "MP": "Mauritius",
    "MT": "Malta",
    "MY": "Malaysia",
    "NI": "Nigeria",
    "NL": "Netherlands",
    "NO": "Norway",
    "NP": "Nepal",
    "NZ": "New Zealand",
    "PE": "Peru",
    "PK": "Pakistan",
    "PL": "Poland",
    "PO": "Portugal",
    "RO": "Romania",
    "RP": "Philippines",
    "RS": "Russia",
    "SF": "South Africa",
    "SG": "Senegal",
    "SI": "Slovenia",
    "SN": "Singapore",
    "SP": "Spain",
    "SW": "Sweden",
    "SZ": "Switzerland",
    "TH": "Thailand",
    "TU": "Turkey",
    "TW": "Taiwan",
    "TZ": "Tanzania",
    "UG": "Uganda",
    "UK": "United Kingdom",
    "UP": "Ukraine",
    "US": "United States",
    "UV": "Burkina Faso",
    "VM": "Vietnam",
    "ZI": "Zimbabwe",
}

CUSTOM_STOPWORDS = {
    "amp", "article", "articles", "bbc", "breaking", "cnn", "content",
    "gallery", "html", "latest", "live", "new", "news", "photo",
    "podcast", "rcna", "report", "reports", "reuters", "said", "say",
    "says", "stories", "story", "three", "two", "update", "updates",
    "video", "videos", "wcm",
}

SLUG_BLACKLIST = {"amp", "article", "articles", "content", "news", "video", "videos", "wcm"}

UUIDISH_RE = re.compile(r"^[0-9a-f]{8,}(?:-[0-9a-f]{4,})+$", re.IGNORECASE)
HEXISH_RE = re.compile(r"^[a-f0-9-]{16,}$", re.IGNORECASE)


def select_country_name(code: str, geo_counter: Counter, fallback_counter: Counter):
    # Choose a country name for a GDELT code.
    if code in GDELT_COUNTRY_OVERRIDES:
        return GDELT_COUNTRY_OVERRIDES[code]

    for candidate, _ in geo_counter.most_common():
        normalized = candidate.lower().strip()
        if normalized and normalized not in GENERIC_COUNTRY_NAMES and len(normalized) > 3:
            return candidate

    for candidate, _ in fallback_counter.most_common():
        normalized = candidate.lower().strip()
        if normalized and normalized not in GENERIC_COUNTRY_NAMES and len(normalized) > 3:
            return candidate

    return pd.NA


def build_country_lookup(location_series: pd.Series) -> dict:
    # Build a lookup from GDELT code to country name.
    geo_candidates = defaultdict(Counter)
    fallback_candidates = defaultdict(Counter)

    for value in location_series.dropna():
        for raw_location in str(value).split(";"):
            parts = raw_location.split("#")
            if len(parts) < 4:
                continue

            label = str(parts[1]).strip()
            country_code = str(parts[2]).strip().upper()

            if len(country_code) != 2 or not country_code.isalpha():
                continue

            fallback_name = label.split(",")[-1].strip()
            if fallback_name:
                fallback_candidates[country_code][fallback_name] += 1

            if "," in label:
                geo_name = label.split(",")[-1].strip()
                if geo_name:
                    geo_candidates[country_code][geo_name] += 1

    lookup = {}
    all_codes = set(fallback_candidates) | set(geo_candidates) | set(GDELT_COUNTRY_OVERRIDES)
    for code in all_codes:
        lookup[code] = select_country_name(code, geo_candidates[code], fallback_candidates[code])
    return lookup


def parse_primary_location(value):
    # Extract the first valid country code from V2Locations.
    if pd.isna(value) or not value:
        return pd.Series({"country_code": pd.NA, "country_name_raw": pd.NA})

    for raw_location in str(value).split(";"):
        parts = raw_location.split("#")
        if len(parts) < 4:
            continue

        country_code = str(parts[2]).strip().upper()
        if len(country_code) == 2 and country_code.isalpha():
            label = str(parts[1]).strip()
            raw_name = label.split(",")[-1].strip() if label else pd.NA
            return pd.Series({"country_code": country_code, "country_name_raw": raw_name})

    return pd.Series({"country_code": pd.NA, "country_name_raw": pd.NA})


def extract_document_tone(value):
    # The first value in V2Tone is the document tone.
    if pd.isna(value) or not value:
        return np.nan

    first_value = str(value).split(",")[0].strip()
    try:
        return float(first_value)
    except ValueError:
        return np.nan


def extract_slug_text(url):
    # Extract a readable slug from the URL path.
    if pd.isna(url) or not url:
        return None

    try:
        parsed_path = unquote(urlparse(str(url)).path)
    except Exception:
        return None

    segments = [segment for segment in parsed_path.split("/") if segment]
    if not segments:
        return None

    candidate = None
    for segment in reversed(segments):
        normalized = unicodedata.normalize("NFKD", segment).encode("ascii", "ignore").decode("ascii")
        cleaned = re.sub(r"\.[a-z0-9]{2,5}$", "", normalized.lower()).strip()

        if not cleaned or cleaned in SLUG_BLACKLIST:
            continue
        if UUIDISH_RE.fullmatch(cleaned) or HEXISH_RE.fullmatch(cleaned):
            continue
        if cleaned.startswith(("article_", "article-")):
            tail = re.sub(r"^article[_-]", "", cleaned)
            if UUIDISH_RE.fullmatch(tail) or HEXISH_RE.fullmatch(tail):
                continue

        alpha_only = re.sub(r"[^a-z]", "", cleaned)
        if len(alpha_only) < 4:
            continue

        candidate = cleaned
        break

    if candidate is None:
        return None

    candidate = re.sub(r"[-_]+", " ", candidate)
    candidate = re.sub(r"\b(rcna|wcm|epi|vnp|html|aspx)\b", " ", candidate)
    candidate = re.sub(r"\d+", " ", candidate)
    candidate = re.sub(r"[^a-z\s]", " ", candidate)
    candidate = re.sub(r"\s+", " ", candidate).strip()
    return candidate or None


def build_stopwords():
    # Combine sklearn stopwords with NLTK stopwords when available.
    stopword_set = set(ENGLISH_STOP_WORDS) | CUSTOM_STOPWORDS

    if NLTK_AVAILABLE:
        try:
            nltk.data.find("corpora/stopwords")
        except LookupError:
            nltk.download("stopwords", quiet=True)

        try:
            stopword_set |= set(stopwords.words("english"))
        except LookupError:
            pass

    return stopword_set


STOPWORDS = build_stopwords()


def tokenize_slug(slug_text):
    # Tokenize slug text into keywords.
    if slug_text is None or pd.isna(slug_text):
        return []

    tokens = []
    for token in slug_text.split():
        token = token.strip().lower()
        if len(token) < 3 or token in STOPWORDS:
            continue
        tokens.append(token)
    return tokens


In [ ]:
country_lookup = build_country_lookup(df_raw["V2Locations"])
parsed_locations = df["V2Locations"].apply(parse_primary_location)

df[["country_code", "country_name_raw"]] = parsed_locations
df["country_name"] = df["country_code"].map(country_lookup).fillna(df["country_name_raw"])
df["date"] = pd.to_datetime(
    df["DATE"].astype("string"),
    format="%Y%m%d%H%M%S",
    errors="coerce",
)
df["sentiment_score"] = df["V2Tone"].apply(extract_document_tone)
df["Themes"] = df["Themes"].fillna("")
df["theme_list"] = df["Themes"].str.split(";").apply(
    lambda items: [item.strip() for item in items if isinstance(item, str) and item.strip()]
)
df["url_slug"] = df["DocumentIdentifier"].apply(extract_slug_text)
df["keyword_list"] = df["url_slug"].apply(tokenize_slug)

# Drop rows only when a field is required for the analysis.
df_clean = df.dropna(
    subset=["date", "sentiment_score", "DocumentIdentifier", "SourceCommonName"]
).copy()
df_geo = df_clean.dropna(subset=["country_code", "country_name"]).copy()

quality_summary = pd.DataFrame(
    {
        "metric": [
            "sample_rows",
            "rows_after_core_cleaning",
            "rows_available_for_geo_analysis",
            "country_parse_rate_on_sample",
            "geo_retention_after_core_cleaning",
        ],
        "value": [
            len(df),
            len(df_clean),
            len(df_geo),
            df["country_code"].notna().mean(),
            len(df_geo) / len(df_clean),
        ],
    }
)

display(quality_summary)
df_clean.head(3)


<div class="alert alert-success">
Cleaning keeps `10,000` of `10,000` sampled rows after the main date and tone parsing steps. For geography, `8,029` rows remain, and the location parser extracts a country code for about **80.3%** of the sample.
</div>


# Part 4: Univariate Analysis (Variation)

We first establish three baselines:

- how sentiment is distributed overall
- how article volume changes through time
- which countries dominate the sampled coverage


## 4.1 Sentiment Baseline

This plot shows the distribution of `sentiment_score`.

<div class="alert alert-info">
This checks whether the dataset is mostly neutral, negative, or positive.
</div>


In [ ]:
plt.figure(figsize=(11, 5))
sns.histplot(df_clean["sentiment_score"], bins=40, color="#4c78a8", kde=True)
plt.axvline(0, color="black", linestyle="--", linewidth=1.2, label="Neutral")
plt.title("Distribution of GDELT Document Tone")
plt.xlabel("Sentiment score")
plt.ylabel("Article count")
plt.legend()
plt.show()


<div class="alert alert-success">
Summary: the sampled distribution is slightly below neutral, with a mean of **-0.99** and a median of **-0.91**. About **60.0%** of sampled articles are negative and **34.5%** are positive. A diverging scale centered at zero fits this distribution.
</div>


## 4.2 Article Volume Over Time

This plot shows how many sampled articles appear each day.

<div class="alert alert-info">
This checks whether coverage is steady or concentrated in a few periods.
</div>


In [ ]:
daily_volume = (
    df_clean.set_index("date")
    .resample("D")
    .agg(article_volume=("DocumentIdentifier", "size"))
    .reset_index()
)
daily_volume["rolling_volume_7d"] = daily_volume["article_volume"].rolling(7, min_periods=3).mean()

plt.figure(figsize=(13, 5))
sns.lineplot(data=daily_volume, x="date", y="article_volume", alpha=0.35, label="Daily count")
sns.lineplot(data=daily_volume, x="date", y="rolling_volume_7d", linewidth=2.5, color="#f58518", label="7-day rolling mean")
plt.title("Article volume through time")
plt.xlabel("Date")
plt.ylabel("Number of sampled articles")
plt.show()


<div class="alert alert-success">
Summary: coverage is uneven through time. The busiest day in the sample is **2023-10-02** with **233** articles.
</div>


## 4.3 Geographic Concentration

This chart shows which countries appear most often in the parsed location field.

<div class="alert alert-info">
This checks whether coverage is spread across countries or concentrated in a few places.
</div>


In [ ]:
country_volume = (
    df_geo["country_name"]
    .value_counts()
    .head(15)
    .rename_axis("country_name")
    .reset_index(name="article_count")
)

plt.figure(figsize=(11, 7))
sns.barplot(data=country_volume, y="country_name", x="article_count", palette="crest")
plt.title("Top 15 countries by sampled article volume")
plt.xlabel("Article count")
plt.ylabel("")
plt.show()


<div class="alert alert-success">
Summary: coverage is concentrated in a few countries. The top countries are **United States (3,457), United Kingdom (594), India (472)**.
</div>


# Part 5: Bivariate and Spatial Analysis (Covariation)

With the baselines established, we now connect variables:

- sentiment with time
- volume with geography
- sentiment with geography


## 5.1 Sentiment Over Time

This plot shows daily sentiment and a 30-day rolling average.

<div class="alert alert-info">
This checks whether sentiment changes over time in a stable way.
</div>


In [ ]:
daily_sentiment = (
    df_clean.set_index("date")
    .resample("D")
    .agg(mean_sentiment=("sentiment_score", "mean"))
    .reset_index()
)
daily_sentiment["rolling_sentiment_30d"] = daily_sentiment["mean_sentiment"].rolling(30, min_periods=7).mean()

plt.figure(figsize=(13, 5))
sns.lineplot(data=daily_sentiment, x="date", y="mean_sentiment", alpha=0.25, label="Daily mean sentiment")
sns.lineplot(
    data=daily_sentiment,
    x="date",
    y="rolling_sentiment_30d",
    linewidth=2.5,
    color="#e45756",
    label="30-day rolling mean",
)
plt.axhline(0, color="black", linestyle="--", linewidth=1)
plt.title("Smoothed sentiment trajectory over time")
plt.xlabel("Date")
plt.ylabel("Mean sentiment score")
plt.show()


<div class="alert alert-success">
Summary: the rolling average changes over the year. The lowest 30-day value is near **2023-01-17** at **-1.59**, and the highest is near **2023-10-08** at **-0.11**.
</div>


## 5.2 Volume by Geography

This map shows article volume by country.

<div class="alert alert-info">
This checks where coverage is dense and where it is sparse.
</div>


In [ ]:
country_volume_map = (
    df_geo.groupby("country_name")
    .agg(
        article_count=("DocumentIdentifier", "size"),
        avg_sentiment=("sentiment_score", "mean"),
    )
    .reset_index()
)

fig = px.choropleth(
    country_volume_map,
    locations="country_name",
    locationmode="country names",
    color="article_count",
    hover_name="country_name",
    hover_data={
        "article_count": ":,",
        "avg_sentiment": ":.2f",
    },
    color_continuous_scale="Viridis",
    title="Publication volume by country",
)
fig.update_layout(template="plotly_dark", margin=dict(l=0, r=0, t=60, b=0))
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))


<div class="alert alert-success">
Summary: the map shows strong clustering, with the largest volume in **United States**.
</div>


## 5.3 Sentiment by Geography

This chart compares average sentiment by country after filtering to countries with enough sampled articles.

<div class="alert alert-info">
This checks which countries are more negative or more positive on average.
</div>


In [ ]:
country_sentiment = (
    df_geo.groupby("country_name")
    .agg(
        article_count=("DocumentIdentifier", "size"),
        avg_sentiment=("sentiment_score", "mean"),
    )
    .query("article_count >= 40")
    .sort_values("avg_sentiment")
)

sentiment_extremes = pd.concat(
    [
        country_sentiment.head(5).assign(group="Most negative"),
        country_sentiment.tail(5).assign(group="Most positive"),
    ]
).sort_values("avg_sentiment")

plt.figure(figsize=(10, 6))
sns.barplot(
    data=sentiment_extremes,
    y="country_name",
    x="avg_sentiment",
    hue="group",
    dodge=False,
    palette={"Most negative": "#e45756", "Most positive": "#54a24b"},
)
plt.axvline(0, color="black", linestyle="--", linewidth=1)
plt.title("Top 5 most negative vs. most positive countries")
plt.xlabel("Average sentiment score")
plt.ylabel("")
plt.legend(title="")
plt.show()


<div class="alert alert-success">
Summary: sentiment varies by country. The most negative countries include **Brazil (-3.20), Russia (-2.17), Israel (-2.10), South Africa (-1.74), China (-1.51)**, while the most positive include **Hungary (4.13), Sweden (3.24), Kenya (0.08), Ghana (-0.14), Ireland (-0.34)**.
</div>


# Part 6: Text and NLP Analysis

GDELT gives us article URLs and themes, which means we can move beyond "where" and "when" into "what kinds of stories are being told?"


## 6.1 URL Slug Keyword Extraction

This chart uses URL slugs to extract common keywords.

<div class="alert alert-info">
This checks which words appear most often in article URLs.
</div>


In [ ]:
keyword_counts = Counter(token for tokens in df_clean["keyword_list"] for token in tokens)
keyword_counts = (
    pd.Series(keyword_counts)
    .sort_values(ascending=False)
    .head(20)
    .rename_axis("keyword")
    .reset_index(name="count")
)

plt.figure(figsize=(11, 7))
sns.barplot(data=keyword_counts, y="keyword", x="count", palette="mako")
plt.title("Most frequent keywords extracted from article URL slugs")
plt.xlabel("Keyword frequency")
plt.ylabel("")
plt.show()


<div class="alert alert-success">
Summary: the most common slug keywords include **vaccine (5,185), covid (3,673), vaccines (2,026), vaccination (1,452), mrna (640)**.
</div>


## 6.2 Thematic Co-occurrence Around "Social Media"

This chart counts the themes that appear most often with social-related themes.

<div class="alert alert-info">
This checks which topics tend to appear with social-related coverage.
</div>


In [ ]:
social_mask = df_clean["theme_list"].apply(lambda themes: any("SOCIAL" in theme.upper() for theme in themes))
social_rows = df_clean.loc[social_mask].copy()

cooccurring_themes = (
    social_rows[["theme_list"]]
    .explode("theme_list")
    .dropna(subset=["theme_list"])["theme_list"]
)
cooccurring_themes = (
    cooccurring_themes[~cooccurring_themes.str.contains("SOCIAL", case=False, na=False)]
    .value_counts()
    .head(10)
    .rename_axis("theme")
    .reset_index(name="count")
)

plt.figure(figsize=(12, 6))
sns.barplot(data=cooccurring_themes, y="theme", x="count", palette="flare")
plt.title("Top 10 themes co-occurring with social-related themes")
plt.xlabel("Co-occurrence count")
plt.ylabel("")
plt.show()


<div class="alert alert-success">
Summary: the most common co-occurring themes are **WB_621_HEALTH_NUTRITION_AND_POPULATION (1,951), UNGP_HEALTHCARE (1,942), GENERAL_HEALTH (1,939), TAX_FNCACT (1,934), HEALTH_VACCINATION (1,920)**.
</div>


## 6.3 Keyword Sentiment Associations

This chart links keywords to average sentiment.

<div class="alert alert-info">
This checks which words are more common in negative or positive articles.
</div>


In [ ]:
keyword_sentiment = (
    df_clean[["sentiment_score", "keyword_list"]]
    .explode("keyword_list")
    .dropna(subset=["keyword_list"])
    .groupby("keyword_list")
    .agg(
        article_count=("keyword_list", "size"),
        avg_sentiment=("sentiment_score", "mean"),
    )
)
keyword_sentiment = keyword_sentiment.query("article_count >= 20")

keyword_extremes = pd.concat(
    [
        keyword_sentiment.nsmallest(10, "avg_sentiment").assign(group="More negative"),
        keyword_sentiment.nlargest(10, "avg_sentiment").assign(group="More positive"),
    ]
).reset_index().rename(columns={"keyword_list": "keyword"})

plt.figure(figsize=(11, 7))
sns.barplot(
    data=keyword_extremes.sort_values("avg_sentiment"),
    y="keyword",
    x="avg_sentiment",
    hue="group",
    dodge=False,
    palette={"More negative": "#e45756", "More positive": "#54a24b"},
)
plt.axvline(0, color="black", linestyle="--", linewidth=1)
plt.title("Keywords associated with negative vs. positive reporting")
plt.xlabel("Average sentiment score")
plt.ylabel("")
plt.legend(title="")
plt.show()


<div class="alert alert-success">
Summary: some keywords are tied to more negative reporting, while others appear in more positive reporting.
</div>


# Part 7: Narrative Affinity Network

This section uses a **Narrative Affinity Network**. The goal is:

> **Which countries are close because they share the same main themes?**

Countries can be far apart geographically but still share similar themes.

<div class="alert alert-info">
Method: keep the most common countries and themes, build a country-theme matrix, and connect countries that share many themes.
</div>


In [ ]:
network_rows = (
    df_geo[["country_name", "sentiment_score", "DocumentIdentifier", "theme_list"]]
    .explode("theme_list")
    .dropna(subset=["theme_list"])
    .rename(columns={"theme_list": "theme"})
)
network_rows = network_rows[network_rows["theme"].ne("")]

top_network_countries = df_geo["country_name"].value_counts().head(12).index.tolist()
top_network_themes = network_rows["theme"].value_counts().head(35).index.tolist()

country_theme_matrix = (
    network_rows[
        network_rows["country_name"].isin(top_network_countries)
        & network_rows["theme"].isin(top_network_themes)
    ]
    .assign(present=1)
    .pivot_table(
        index="country_name",
        columns="theme",
        values="present",
        aggfunc="max",
        fill_value=0,
    )
    .reindex(index=top_network_countries, fill_value=0)
)

shared_theme_matrix = country_theme_matrix.dot(country_theme_matrix.T).copy()
for idx in range(len(shared_theme_matrix)):
    shared_theme_matrix.iat[idx, idx] = 0

edge_candidates = []
edge_weights = []
for idx, country_a in enumerate(top_network_countries):
    for country_b in top_network_countries[idx + 1 :]:
        weight = int(shared_theme_matrix.loc[country_a, country_b])
        if weight > 0:
            edge_candidates.append((country_a, country_b, weight))
            edge_weights.append(weight)

edge_threshold = max(3, int(np.quantile(edge_weights, 0.70))) if edge_weights else 1

G = nx.Graph()

country_metrics = (
    df_geo.groupby("country_name")
    .agg(
        article_count=("DocumentIdentifier", "size"),
        avg_sentiment=("sentiment_score", "mean"),
    )
    .reindex(top_network_countries)
)

for country in top_network_countries:
    G.add_node(
        country,
        article_count=country_metrics.loc[country, "article_count"],
        avg_sentiment=country_metrics.loc[country, "avg_sentiment"],
    )

for country_a, country_b, weight in edge_candidates:
    if weight >= edge_threshold:
        G.add_edge(country_a, country_b, weight=weight)

if G.number_of_edges() == 0:
    for country_a, country_b, weight in edge_candidates[:10]:
        G.add_edge(country_a, country_b, weight=weight)

pos = nx.spring_layout(G, seed=42, k=1.1)

node_sizes = [G.nodes[node]["article_count"] * 2.8 for node in G.nodes]
node_colors = [G.nodes[node]["avg_sentiment"] for node in G.nodes]
edge_widths = [G.edges[edge]["weight"] * 0.18 for edge in G.edges]

plt.figure(figsize=(12, 9))
nx.draw_networkx_edges(G, pos, width=edge_widths, edge_color="#94a3b8", alpha=0.35)
nodes = nx.draw_networkx_nodes(
    G,
    pos,
    node_size=node_sizes,
    node_color=node_colors,
    cmap=plt.cm.coolwarm,
    edgecolors="white",
    linewidths=0.8,
)
nx.draw_networkx_labels(G, pos, font_size=10)
plt.colorbar(nodes, label="Average sentiment score")
plt.title("Narrative affinity network: countries linked by shared dominant themes")
plt.axis("off")
plt.show()


# Part 8: Other Key Insights


In [ ]:
implementation_summary = pd.DataFrame(
    {
        "metric": [
            "Absolute minimum sentiment",
            "Absolute maximum sentiment",
            "5th percentile sentiment",
            "95th percentile sentiment",
            "95th percentile country article volume",
        ],
        "value": [
            df_clean["sentiment_score"].min(),
            df_clean["sentiment_score"].max(),
            df_clean["sentiment_score"].quantile(0.05),
            df_clean["sentiment_score"].quantile(0.95),
            df_geo["country_name"].value_counts().quantile(0.95),
        ],
    }
)

implementation_summary
